In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [81]:
df = pd.read_json(
    "../data/threat_intel_dataset.jsonl",
    lines=True
)

df.head()

,instruction,input,output
0,You are a cybersecurity threat intelligence an...,CVE ID: CVE-2001-1481\nDescription: Xitami 2.4...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...
1,You are a cybersecurity threat intelligence an...,CVE ID: CVE-1999-0426\nDescription: The defaul...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...
2,You are a cybersecurity threat intelligence an...,CVE ID: CVE-1999-1324\nDescription: VAXstation...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...
3,You are a cybersecurity threat intelligence an...,CVE ID: CVE-2000-1218\nDescription: The defaul...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...
4,You are a cybersecurity threat intelligence an...,CVE ID: CVE-2000-0944\nDescription: CGI Script...,THREAT INTELLIGENCE REPORT\n\nSEVERITY: CRITIC...


In [82]:
print(f"Shape: {df.shape}")
print(f"Info: {df.info()}")
print(f"Describe: {df.describe()}")

Shape: (472, 3)
<class 'pandas.DataFrame'>
RangeIndex: 472 entries, 0 to 471
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   instruction  472 non-null    str  
 1   input        472 non-null    str  
 2   output       472 non-null    str  
dtypes: str(3)
memory usage: 11.2 KB
Info: None
Describe:                                               instruction  \
count                                                 472   
unique                                                  2   
top     You are a cybersecurity threat intelligence an...   
freq                                                  471   

                                                    input  \
count                                                 472   
unique                                                472   
top     CVE ID: CVE-2001-1481\nDescription: Xitami 2.4...   
freq                                                    1   

                     

In [83]:
print(df.loc[0, "instruction"])
print(df.loc[0, "input"])
print(df.loc[0, "output"][:1000])

You are a cybersecurity threat intelligence analyst. Analyze the following vulnerability data and generate a comprehensive, structured Threat Intelligence Report with severity assessment, technical analysis, MITRE ATT&CK mapping, remediation steps, and detection rules.
CVE ID: CVE-2001-1481
Description: Xitami 2.4 through 2.5 b4 stores the Administrator password in plaintext in the default.aut file, whose default permissions are world-readable, which allows remote attackers to gain privileges.
CVSS Score: 9.8
Severity: CRITICAL
Affected Products: cpe:2.3:a:xitami:xitami:*:*:*:*:*:*:*:*, cpe:2.3:a:xitami:xitami:2.5:beta4:*:*:*:*:*:*
References: http://archives.neohapsis.com/archives/win2ksecadvice/2000-q4/0109.html, http://www.securityfocus.com/archive/1/242375, http://www.securityfocus.com/bid/3582, https://exchange.xforce.ibmcloud.com/vulnerabilities/7600, http://archives.neohapsis.com/archives/win2ksecadvice/2000-q4/0109.html
THREAT INTELLIGENCE REPORT

SEVERITY: CRITICAL (CVSS 9.8)


Threat Categories:
- Remote Code Execution
- SQL Injection
- Cross-Site Scripting
- Credential Exposure
- Privilege Escalation
- Denial of Service
- Information Disclosure
- Directory Traversal
- Authentication Bypass
- Unknown

In [32]:
df.isnull().sum()

instruction    0
input          0
output         0
dtype: int64

In [33]:
for i in range(5):
    print("ROW:", i)
    print(df.loc[i, "input"])
    print("-" * 80)

ROW: 0
CVE ID: CVE-2001-1481
Description: Xitami 2.4 through 2.5 b4 stores the Administrator password in plaintext in the default.aut file, whose default permissions are world-readable, which allows remote attackers to gain privileges.
CVSS Score: 9.8
Severity: CRITICAL
Affected Products: cpe:2.3:a:xitami:xitami:*:*:*:*:*:*:*:*, cpe:2.3:a:xitami:xitami:2.5:beta4:*:*:*:*:*:*
References: http://archives.neohapsis.com/archives/win2ksecadvice/2000-q4/0109.html, http://www.securityfocus.com/archive/1/242375, http://www.securityfocus.com/bid/3582, https://exchange.xforce.ibmcloud.com/vulnerabilities/7600, http://archives.neohapsis.com/archives/win2ksecadvice/2000-q4/0109.html
--------------------------------------------------------------------------------
ROW: 1
CVE ID: CVE-1999-0426
Description: The default permissions of /dev/kmem in Linux versions before 2.0.36 allows IP spoofing.
CVSS Score: 9.8
Severity: CRITICAL
Affected Products: cpe:2.3:o:suse:suse_linux:6.0:*:*:*:*:*:*:*
References:

# Threat Detection
Takes somesort of raw input/data and determines if it is a threat
Am I affected?
How am I affected?

In [126]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

class ThreatClassificationAgent:
    def __init__(self):
        self.threats = {
    "SQL Injection": "vulnerability allowing attackers to manipulate SQL database queries through unsanitised user input",
    "XSS": "cross-site scripting vulnerability involving malicious script injection and execution in a user's browser",
    "Denial of Service": "vulnerability allowing attackers to exhaust resources, crash systems, or disrupt service availability",
    "Authentication Bypass": "vulnerability allowing attackers to gain unauthorised access without valid authentication credentials",
    "Privilege Escalation": "vulnerability allowing attackers to gain elevated permissions or administrative access",
    "Remote Code Execution": "vulnerability allowing attackers to remotely execute arbitrary code or system commands on a target machine"
}

    def classify(self, cve_text):
        input_embeddings = model.encode(cve_text)
        output_embeddings = model.encode(list(self.threats.values()))

        input_embeddings = input_embeddings.reshape(1,-1)
      
        similarity = cosine_similarity(input_embeddings, output_embeddings)
        one_row = similarity[0]
        idx = one_row.argmax()
        e = list(self.threats.keys())
        return e[idx]
    
    def __str__(self):
        return "Threat Classification Agent"

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4766.10it/s]


In [124]:
agent = ThreatClassificationAgent()

x = 4
result = agent.classify(df.loc[x, "input"])
print(result)

Authentication Bypass


In [125]:
agent = ThreatClassificationAgent()

tests = [
    "The application allows attackers to execute arbitrary code remotely.",
    "The login form is vulnerable to SQL injection through unsanitised input.",
    "A reflected cross-site scripting vulnerability allows script execution.",
    "The server crashes when receiving malformed packets, causing denial of service.",
    "Plaintext passwords are stored in a world-readable file."
]

for text in tests:
    print(text)
    print(agent.classify(text))
    print()

The application allows attackers to execute arbitrary code remotely.
Remote Code Execution

The login form is vulnerable to SQL injection through unsanitised input.
SQL Injection

A reflected cross-site scripting vulnerability allows script execution.
XSS

The server crashes when receiving malformed packets, causing denial of service.
Denial of Service

Plaintext passwords are stored in a world-readable file.
Authentication Bypass



# Severity Agent


In [ ]:
class SeverityAgent:
    def __init__(self):
        pass

# Mitigation Agents 

In [ ]:
class MitigationAgent:
     def __init__(self):
        pass

# Coordinator Agent

In [ ]:
class CoordinatorAgent:
    def __init__(self):
        pass